# 详细文档: 2D模型 - 非结构化网格

**相关模块:** `model_2d/mesh.py`

## 目的
此文档旨在详细介绍2D水动力模型所使用的核心数据结构——非结构化三角网格（Unstructured Triangular Mesh）。有限体积法（Finite Volume Method）的2D模型是在一系列不重叠的控制体积（或称单元、`Face`）上求解守恒定律的，这些控制体积共同构成了计算区域。理解网格的构建和拓扑关系是理解2D模型工作原理的基础。

此笔记本将：
1.  解释构成网格的基本元素：`Node`（节点）、`Edge`（边）和`Face`（单元）。
2.  演示如何从最基础的点和三角形列表，使用`Mesh.build_from_points_and_triangles`方法来构建一个完整的、具有拓扑关系的网格。
3.  解释网格构建过程中如何自动识别内部边和边界边。
4.  通过详细的可视化，展示一个完整网格的所有组成部分。

## 1. 环境设置

我们导入所需的库和`mesh.py`模块。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from model_2d.mesh import Mesh

## 2. 定义基础几何

构建一个网格，我们只需要两样最基本的信息：
1.  **点 (Points)**: 一个包含所有节点`(x, y)`坐标的列表。
2.  **三角形 (Triangles)**: 一个定义了所有三角形单元的列表。每个三角形由三个整数索引组成，这些索引指向`points`列表中的对应节点。

我们以一个简单的矩形区域为例，它由两个三角形组成。

In [ ]:
# 4个节点定义一个矩形
points = [
    (0, 0), # 0
    (2, 0), # 1
    (2, 1), # 2
    (0, 1)  # 3
]

# 2个三角形组成这个矩形
triangles = [
    (0, 1, 2), # Face 0
    (0, 2, 3)  # Face 1
]

## 3. 构建完整网格

现在，我们创建一个`Mesh`实例，并调用`build_from_points_and_triangles`方法。这个方法会完成所有繁重的工作：
- 创建`Node`对象和`Face`对象。
- 遍历所有`Face`的边，使用一个字典来查找共享相同节点的边，从而自动创建`Edge`对象。
- 识别出只被一个`Face`拥有的边，并将它们标记为**边界边** (`boundary_edges`)。
- 建立`Face`和`Edge`之间的双向连接。

In [ ]:
mesh = Mesh()
mesh.build_from_points_and_triangles(points, triangles)

print("--- 网格拓扑信息 ---")
print(f"总节点数: {len(mesh.nodes)}")
print(f"总单元数 (Faces): {len(mesh.faces)}")
print(f"总边数 (Edges): {len(mesh.edges)}")
print(f"边界边数: {len(mesh.boundary_edges)}")

# 检查一条内部边和一条边界边
internal_edge = None
for edge in mesh.edges:
    if edge.face1 is not None and edge.face2 is not None:
        internal_edge = edge
        break

print(f"\n内部边 {internal_edge.id} (连接节点 {internal_edge.nodes[0].id} 和 {internal_edge.nodes[1].id}):")
print(f"  - 左侧单元: Face {internal_edge.face1.id}")
print(f"  - 右侧单元: Face {internal_edge.face2.id}")

boundary_edge = mesh.boundary_edges[0]
print(f"\n边界边 {boundary_edge.id} (连接节点 {boundary_edge.nodes[0].id} 和 {boundary_edge.nodes[1].id}):")
print(f"  - 左侧单元: Face {boundary_edge.face1.id}")
print(f"  - 右侧单元: {boundary_edge.face2}")

## 4. 可视化网格

最后，我们编写一个绘图函数来可视化我们创建的网格，并在图上标注出所有的节点、单元中心、边和边界边，以加深理解。

In [ ]:
def plot_mesh(mesh):
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # 绘制所有边
    for edge in mesh.edges:
        x_coords = [edge.nodes[0].x, edge.nodes[1].x]
        y_coords = [edge.nodes[0].y, edge.nodes[1].y]
        ax.plot(x_coords, y_coords, 'k-', linewidth=1, alpha=0.5)
        
    # 高亮边界边
    for edge in mesh.boundary_edges:
        x_coords = [edge.nodes[0].x, edge.nodes[1].x]
        y_coords = [edge.nodes[0].y, edge.nodes[1].y]
        ax.plot(x_coords, y_coords, 'r-', linewidth=3, label='Boundary Edge' if 'Boundary Edge' not in ax.get_legend_handles_labels()[1] else "")
        
    # 绘制节点
    node_x = [n.x for n in mesh.nodes]
    node_y = [n.y for n in mesh.nodes]
    ax.plot(node_x, node_y, 'bo', label='Nodes')
    for node in mesh.nodes:
        ax.text(node.x + 0.05, node.y + 0.05, f'N{node.id}', fontsize=12)
        
    # 绘制单元中心
    face_x = [f.centroid[0] for f in mesh.faces]
    face_y = [f.centroid[1] for f in mesh.faces]
    ax.plot(face_x, face_y, 'g*', markersize=10, label='Face Centroids')
    for face in mesh.faces:
        ax.text(face.centroid[0], face.centroid[1] - 0.1, f'F{face.id}', fontsize=12, ha='center')
        
    ax.set_title('Visualization of Mesh Topology')
    ax.set_xlabel('X coordinate')
    ax.set_ylabel('Y coordinate')
    ax.set_aspect('equal', 'box')
    ax.legend()
    ax.grid(True)
    plt.show()

plot_mesh(mesh)